# SFT Training — Qwen2.5-0.5B on databricks-dolly-15k

Full fine-tuning (no LoRA yet — that starts Week 3) using `trl.SFTTrainer`. Runs a small dry run first to catch out-of-memory errors before committing to the full training time, per Week 1's objective.

**Before running:** on Kaggle, enable **Settings → Accelerator → GPU T4 x2**. Do not pick P100 — modern PyTorch wheels have dropped kernel support for the P100's older `sm_60` architecture.

## 0. Restrict to a single GPU

Kaggle's "T4 x2" gives two GPUs in the same session. Hugging Face's `Trainer` auto-detects more than one visible GPU and wraps the model in `torch.nn.DataParallel` to split batches across both — but our code only moves the model to one device (`cuda:0`), so the two disagree and crash with a device-mismatch error the first time a batch runs. A 0.5B model doesn't need two GPUs anyway (multi-GPU training is out of scope for Week 1), so the simplest fix is to make only one GPU visible at all, before `torch` is even imported.

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

## 1. Pull the repo and install dependencies

This repo is private. Before running:

1. Create a GitHub Personal Access Token (fine-grained, scoped to just this repo, **Contents: Read-only**).
2. In this Kaggle notebook: **Add-ons → Secrets → Add a new secret**, name it `GITHUB_TOKEN`.
3. Run the cell below — it reads the secret and never prints it.

**Sparse checkout, on purpose:** a full clone would also pull the Obsidian notes (`00_Admin_&_Roadmap/`, etc.), some of which have `&` in their filename — harmless for training, but it broke Kaggle's "New Dataset from Output" later (it scans everything under `/kaggle/working` and rejects that character). Only `src/` (plus the repo-root files, included automatically in `--cone` mode: `config.yaml`, `requirements.txt`, `README.md`) is actually needed here.

In [ ]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
github_token = secrets.get_secret("GITHUB_TOKEN")
repo_url = f"https://{github_token}@github.com/zoom-BT/llm-alignment-internship.git"

!git clone --filter=blob:none --no-checkout {repo_url}
%cd llm-alignment-internship
!git sparse-checkout init --cone
!git sparse-checkout set src
!git checkout main
!pip install -q -r requirements.txt

## 2. Confirm the GPU is visible

In [ ]:
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

## 3. Dry run (120 steps) — check for out-of-memory errors AND that eval/early-stopping actually fire

This is deliberately not run locally: full fine-tuning needs memory for weights + gradients + AdamW's optimizer state (roughly 4-6x the model size), which is exactly the kind of load that segfaulted this laptop during plain inference in BF16. A T4 has 16 GB — comfortable for a 0.5B model, but worth confirming with a short run before spending the full training time.

**120 steps, not 20:** `config.yaml` sets `eval_steps: 100`, so a 20-step dry run never actually triggers an evaluation — it would only prove there's no OOM, not that `eval_strategy`, `load_best_model_at_end`, and `EarlyStoppingCallback` (this week's new code paths) work. 120 > 100 guarantees at least one real evaluation happens.

In [ ]:
import yaml
from src.train import run_sft

config = yaml.safe_load(open("config.yaml"))
trainer = run_sft(config, max_steps=120)
print("Dry run finished without OOM, and eval/early-stopping ran at least once.")

## 4. Full training run

Only run this once the dry run above completed cleanly. `max_steps=-1` (the default) means "ignore step count, run the full `num_epochs` from `config.yaml`".

In [ ]:
trainer = run_sft(config)
print("Training complete. Checkpoint saved to results/checkpoints/final")

## 4a. Display the training curves

`run_sft` already saved `results/training_curve.png` (train + eval loss) and `results/training_log_history.json` (the raw numbers) as a side effect of training. Displaying it here means it shows up directly in this cell's output — including in a committed ("Save & Run All") version's Output tab, not just in a live interactive session.

In [ ]:
from IPython.display import Image, display

display(Image(filename="results/training_curve.png"))

## 4b. Zip the checkpoint and clean up

Turns 6 loose files into 1 zip, and removes the intermediate `checkpoint-*` snapshots `Trainer` saves periodically for resuming (large — each one includes optimizer state, not just weights) but that we don't need for evaluation. Makes the next "New Dataset from Output" both possible (fewer/no problematic filenames left, since nothing outside `results/checkpoints/` was pulled in the first place with the sparse clone) and much smaller.

In [ ]:
import shutil
from pathlib import Path

shutil.make_archive("results/checkpoints/final", "zip", "results/checkpoints/final")

for path in Path("results/checkpoints").iterdir():
    if path.name != "final.zip":
        shutil.rmtree(path)

print("Kept only results/checkpoints/final.zip")

## 5. Inspect training curves

Logged locally via TensorBoard (`config['training']['logging_backend']`), no external account needed.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir results/checkpoints

## 6. Next step

The fine-tuned checkpoint now lives at `results/checkpoints/final` in this Kaggle session. The "after" evaluation (comparing against the baseline perplexity of 13.92 from `results/baseline_results.json`) runs in a follow-up notebook, reusing this same session so the checkpoint doesn't need to be re-downloaded — checkpoints are large binaries and are deliberately git-ignored, unlike the small JSON result files.